# Titanic — Feature Engineering

**Purpose:** Turn raw passenger data into features that a machine learning model can use.  
All transformations are motivated by what the EDA revealed — nothing is added arbitrarily.

**Output:** `data/train_engineered.csv` and `data/test_engineered.csv`, ready to be loaded by `04_modeling.ipynb`.

---
**Chain of thought:**
1. Load clean raw data
2. Apply transformations motivated by EDA
3. Add new features: `WomanOrChild`, `FarePerPerson`, `TicketGroupSize`, `AgeGroup`
4. Define the feature lists for the preprocessing pipeline
5. Export engineered datasets

## 0) Setup

In [ ]:
import os
import numpy as np
import pandas as pd

# ── Constants ────────────────────────────────────────────────────────────────
DATA_DIR    = "../data"
OUTPUT_DIR  = "../data"   # engineered files go back into data/
RANDOM_STATE = 42

print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

## 1) Load data

We load the original raw files. No transformation has been applied yet.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("train:", train.shape, " test:", test.shape)
train.head(3)

## 2) Feature engineering

We apply the same transformations to train and test to avoid any mismatch at prediction time.

### Features from EDA (motivated by the patterns we saw)

| Feature | Source | Why |
|---------|--------|-----|
| `Title` | `Name` | Encodes gender + social status in one signal. More predictive than `Sex` alone. |
| `FamilySize` | `SibSp + Parch + 1` | EDA showed a U-shaped survival pattern by family size. |
| `IsAlone` | `FamilySize == 1` | Captures the solo-traveler penalty cleanly. |
| `Deck` | First letter of `Cabin` | Higher decks correlated with higher class and higher survival. |
| `Fare_log1p` | `log1p(Fare)` | Fare is heavily right-skewed; log transform makes it better for linear models. |

### New features (personal additions)

| Feature | Formula | Why |
|---------|---------|-----|
| `WomanOrChild` | `Sex == female OR Age < 15` | Directly encodes the historical rescue protocol — "women and children first". Rather than letting the model infer it, I model it explicitly. |
| `FarePerPerson` | `Fare / FamilySize` | Group tickets split the cost. A solo passenger paying £7 is different from a family of 5 paying £7 total. |
| `TicketGroupSize` | Count of passengers sharing the same ticket | Reveals travel companions beyond declared family (friends, employees, organized groups). |
| `AgeGroup` | Binned Age | Age effect is non-linear. A child of 8 has very different survival odds than a 20-year-old. |

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all feature engineering to a raw DataFrame (train or test)."""
    out = df.copy()

    # ── EDA-motivated features ────────────────────────────────────────────────

    # Title from Name (encodes gender + social status)
    title = out['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
    common = {'Mr', 'Mrs', 'Miss', 'Master'}
    out['Title'] = title.where(title.isin(common), 'Other')

    # Family size and solo-traveler flag
    out['FamilySize'] = out['SibSp'].fillna(0) + out['Parch'].fillna(0) + 1
    out['IsAlone']    = (out['FamilySize'] == 1).astype(int)

    # Deck from Cabin (first letter; NaN → 'U' for Unknown)
    deck = out['Cabin'].astype(str).str[0]
    out['Deck'] = deck.where(deck != 'n', 'U')  # 'nan'[0] == 'n'

    # Log-fare to reduce right-skew
    out['Fare_log1p'] = np.log1p(out['Fare'])

    # ── New personal features ─────────────────────────────────────────────────

    # WomanOrChild — explicit encoding of the rescue protocol
    out['WomanOrChild'] = (
        (out['Sex'] == 'female') | (out['Age'] < 15)
    ).astype('Int8')  # nullable int because Age has NaN

    # FarePerPerson — normalize fare by group size
    out['FarePerPerson'] = out['Fare'] / out['FamilySize']

    # TicketGroupSize — count passengers sharing the same ticket number
    ticket_counts = out.groupby('Ticket')['PassengerId'].transform('count')
    out['TicketGroupSize'] = ticket_counts

    # AgeGroup — bin ages into life stages
    bins   = [0, 12, 18, 35, 60, 100]
    labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
    out['AgeGroup'] = pd.cut(out['Age'], bins=bins, labels=labels)

    return out


train_eng = engineer_features(train)
test_eng  = engineer_features(test)

print("train_eng shape:", train_eng.shape)
print("test_eng  shape:", test_eng.shape)

new_cols = ['Title', 'FamilySize', 'IsAlone', 'Deck', 'Fare_log1p',
            'WomanOrChild', 'FarePerPerson', 'TicketGroupSize', 'AgeGroup']
train_eng[new_cols].head(8)

## 3) Quick sanity checks

Verify that the new features behave as expected before handing off to modeling.

In [ ]:
# WomanOrChild survival validation
# We expect this to be one of the strongest predictors
woc_survival = train_eng.groupby('WomanOrChild')['Survived'].mean()
print("Survival rate — WomanOrChild:")
print(woc_survival.rename({0: 'Man (adult)', 1: 'Woman or Child'}))

print()

# FarePerPerson — compare with raw Fare for a family group
print("FarePerPerson sample (first 5 rows with FamilySize > 1):")
print(train_eng[train_eng['FamilySize'] > 1][['Fare', 'FamilySize', 'FarePerPerson']].head(5).to_string())

print()

# TicketGroupSize — check it finds groups correctly
print("TicketGroupSize value counts (top 5):")
print(train_eng['TicketGroupSize'].value_counts().head())

print()

# AgeGroup distribution
print("AgeGroup value counts:")
print(train_eng['AgeGroup'].value_counts().sort_index())

## 4) Feature lists for modeling

We declare which columns go into the model, split by numeric and categorical.  
These lists are what `04_modeling.ipynb` will import — keeping them here makes the separation clean.

In [ ]:
TARGET = 'Survived'
ID_COL = 'PassengerId'

NUMERIC_FEATURES = [
    'Age', 'SibSp', 'Parch', 'Fare', 'Pclass',
    'FamilySize', 'Fare_log1p', 'FarePerPerson', 'TicketGroupSize'
]

CATEGORICAL_FEATURES = [
    'Sex', 'Embarked', 'Title', 'IsAlone', 'Deck', 'WomanOrChild', 'AgeGroup'
]

print(f"Numeric features  ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
print(f"Total raw features: {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")

## 5) Export engineered datasets

Save to CSV so `04_modeling.ipynb` can load them directly without re-running this notebook.

In [ ]:
train_out_path = os.path.join(OUTPUT_DIR, "train_engineered.csv")
test_out_path  = os.path.join(OUTPUT_DIR, "test_engineered.csv")

train_eng.to_csv(train_out_path, index=False)
test_eng.to_csv(test_out_path,  index=False)

print(f"Saved train_engineered.csv  →  {train_eng.shape}")
print(f"Saved test_engineered.csv   →  {test_eng.shape}")
print()
print("New columns added:", new_cols)